In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataloader import load_test_set
from pathlib import Path

In [ ]:
def data_attr():
    # Piero data\n",
    data_path_array = [
                           '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Piero_20131202.npz',  ##
                           '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Piero_20140109.npz',  ##
                           '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Piero_20140116.npz',  ##
#                            '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Piero_20140606.npz',
#                            '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Piero_20140701.npz',
#                            '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Piero_20140922.npz',
        
#                            '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Cornelio_20140424.npz',  ##
#                            '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Cornelio_20140515.npz',
#                            '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Cornelio_20140520.npz',
#                            '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Cornelio_20140527.npz',  ##
#                            '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Cornelio_20140528.npz',  ##
#                            '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Cornelio_20140529.npz',
#                            '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Cornelio_20140601.npz',
#                            '/raid/home/tubitoal/DMM/dmm/dataset/MUA/data/Cornelio_20140606.npz',
    ]
    
    selection ="((y_stop == True) | ((y_stop == False)&(y_reward == True)))"# selezioni tutto il movimento
    
    return data_path_array, selection

base_path = Path().cwd().parent
plt.style.use(base_path / "paper.mplstyle")

In [ ]:
data_path_array, selection = data_attr()

(data,
RT,
SSD,
direction,
session,
subject) = load_test_set(data_path_array,selection)

mask_ws = (RT > 0) & (SSD > 0)
RT_ws = RT[mask_ws]
SSD_ws = SSD[mask_ws]

mask_stop = SSD > 0
SSD_stop = SSD[mask_stop]

RT_nostop = RT[RT>0]

#intervals = [[60, 65], [80, 85], [100, 105]]#, [120, 125]]#, [140, 145]]
intervals = [[40, 45], [60, 65], [80, 85], [100, 105]]#, [120, 125]]

SSD_list = []
frac_list = []
RT_list = []
SSRT_list = []
for interval in intervals:
    left_bound, right_bound = interval
    n_SSD_ws = np.sum((SSD_ws > left_bound) & (SSD_ws < right_bound))
    n_SSD_stop = np.sum((SSD_stop > left_bound) & (SSD_stop < right_bound))
    print(n_SSD_ws, n_SSD_stop)
    frac_ws = n_SSD_ws/n_SSD_stop
    if frac_ws>0.99:
        frac_ws = 0.96
    RT_val = np.quantile(RT_nostop, frac_ws)
    SSRT_val = RT_val - (right_bound - 2)
    RT_list.append(RT_val)
    SSRT_list.append(SSRT_val)
    SSD_list.append(right_bound - 2)
    frac_list.append(frac_ws)
    
SSRT_array = np.array(SSRT_list)    
print(f"mean SSRT: {SSRT_array.mean()*5:.0f}ms")
    
num_bins = 40
min_value = 60
max_value = 180
bin_edges = np.linspace(min_value, max_value, num_bins + 1)
    
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(SSD_list, frac_list)
# Add labels and title
ax[0].set_xlabel('SSD_ws values')
ax[0].set_ylabel('fraction of ws trials')
ax[0].set_title("Fraction of ws trials w.r.t total stop trials")

ax[1].hist(RT, bins=bin_edges, alpha = 0.7, density = True, color='skyblue', edgecolor='black')
ax[1].axvline(RT_list[0], color="r",linestyle="--",label=f"0")
ax[1].axvline(RT_list[1], color="b",linestyle="--",label=f"1")
ax[1].axvline(RT_list[2], color="g",linestyle="--",label=f"2")

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].scatter(SSD_list, SSRT_list)
# Add labels and title
ax[0].set_xlabel('SSD_ws values')
ax[0].set_ylabel('SSRT')
ax[0].set_title("SSRT vs SSD")

ax[1].scatter(RT_list, SSRT_list)
# Add labels and title
ax[1].set_xlabel('RT values')
ax[1].set_ylabel('SSRT')
ax[1].set_title("SSRT vs RT")

In [ ]:
data_path_array, selection = data_attr()

(data,
RT,
SSD,
direction,
session,
subject) = load_test_set(data_path_array,selection)

mask_ws = (RT > 0) & (SSD > 0)
RT_ws = RT[mask_ws]
SSD_ws = SSD[mask_ws]

mask_stop = SSD > 0
SSD_stop = SSD[mask_stop]

print(np.unique(SSD_ws))
print(np.unique(SSD_stop))

num_bins = 40
min_value = 0
max_value = 200*5
bin_edges = np.linspace(min_value, max_value, num_bins + 1)

fig, ax = plt.subplots()
ax.hist(SSD_ws*5, bins=bin_edges, alpha = 0.5, color='r', edgecolor='black', label = "ws SSD")
ax.hist(SSD_stop*5, bins=bin_edges, alpha = 0.5, color='skyblue', edgecolor='black', label = "SSD")

# Add labels and title
ax.set_xlabel('SSD_ws values')
ax.set_ylabel('% of trials')
ax.set_title("Histograms of wrong stop SSD")
ax.legend()

In [ ]:
data_path_array, selection = data_attr()

(data,
RT,
SSD,
direction,
session,
subject) = load_test_set(data_path_array,selection)

print(data.shape)

# print(data.shape)
data = data[:, ::2, :]

transition = data[:, 1:] - data[:, :-1]
transition_flat = transition.reshape(-1)

num_bins = 100
min_value = -1.5
max_value = 1.5
bin_edges = np.linspace(min_value, max_value, num_bins + 1)

print(f"The transition values have mean {transition_flat.mean():.2f} and std {transition_flat.std():.2f}")

fig, ax = plt.subplots()
ax.hist(transition_flat, bins=bin_edges, alpha = 0.6, density = True, color='skyblue', edgecolor='none')
ax.axvline(transition_flat.mean(), color="k",linestyle="--",label=f"transition mean")
ax.axvline(transition_flat.std(), color="r",linestyle="--",label=f"transition std")

# Add labels and title
ax.set_xlabel('Transition values')
ax.set_ylabel('Counts')
ax.set_xticks([-1, 0, 1])
ax.set_yticks([])

plt.savefig("Transition_values_Cornelio.png")

In [ ]:
data_path_array, selection = data_attr()

(data,
RT,
SSD,
direction,
session,
subject) = load_test_set(data_path_array,selection)

mask = (RT > 0) & (SSD > 0)
RT_ws = RT[mask]
SSD_ws = SSD[mask]

diff = (RT_ws - SSD_ws)*5

RT_strange = RT_ws[diff<5]

num_bins = 30
min_value = 0
max_value = 80*5
bin_edges = np.linspace(min_value, max_value, num_bins + 1)

fig, ax = plt.subplots()
ax.hist(diff, bins=bin_edges, alpha = 1, density = True, color='skyblue', edgecolor='black')
ax.axvline(diff.mean(), color="k",linestyle="--",label=f"diff mean")

# Add labels and title
ax.set_xlabel('RT - SSD difference in wrong stops')
ax.set_ylabel('% of trials')
ax.set_title("Histograms of RT-SSD for wrong stop trials")